# Earthdata Cloud Access and PACE Demo

Based on the PACE Hackweek Tutorial started by Anna Windle and continued by Patrick Gray, slightly modified for Split workshop and v3.2 by Ivona Cetinic

An [Earthdata Login][edl] account is required to access data from the NASA Earthdata system, including NASA ocean color data.

</div>

[edl]: https://urs.earthdata.nasa.gov/
A bit about the [product suits](https://pace.oceansciences.org/data_table.htm) that PACE currently is producing - note that this changes weekly.

We will explore [NASA Worldview](https://worldview.earthdata.nasa.gov/).

Then - we will figure out all different ways to get [PACE](https://pace.oceansciences.org/access_pace_data.htm) (and other ocean color) data. 


All the amazing code to play with PACE data can be found on our [HelpHub](https://nasa.github.io/oceandata-notebooks/index.html). Please explore it after this tutorial. 

## 1. Setup

We begin by importing the packages used in this notebook.

This will also install a few packages in case you're working in colab.

In [ ]:
pip install earthaccess

In [ ]:
pip install cartopy

In [ ]:
pip install cmocean

In [ ]:
import earthaccess
import xarray as xr
from xarray.backends.api import open_datatree
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
from matplotlib.colors import LogNorm
import cmocean
import pandas as pd
from matplotlib.patches import Rectangle
from matplotlib.colors import LogNorm

The last import provides a preview of the `DataTree` object. Once it is fully integrated into XArray,
the additional import won't be needed, as the function will be available as `xr.open_datree`.

[back to top](#Contents)

## 2. NASA Earthdata Authentication

Next, we authenticate using our Earthdata Login
credentials. Authentication is not needed to search publicaly
available collections in Earthdata, but is always needed to access
data. We can use the `login` method from the `earthaccess`
package. This will create an authenticated session when we provide a
valid Earthdata Login username and password. The `earthaccess`
package will search for credentials defined by **environmental
variables** or within a **.netrc** file saved in the home
directory. If credentials are not found, an interactive prompt will
allow you to input credentials.

<div class="alert alert-info" role="alert">
    
The `persist=True` argument ensures any discovered credentials are
stored in a `.netrc` file, so the argument is not necessary (but
it's also harmless) for subsequent calls to `earthaccess.login`.

</div>

In [ ]:
auth = earthaccess.login(persist=True)

[back to top](#Contents)

## 3. Search for Data

Collections on NASA Earthdata are discovered with the
`search_datasets` function, which accepts an `instrument` filter as an
easy way to get started. Each of the items in the list of
collections returned has a "short-name".

In [ ]:
results = earthaccess.search_datasets(instrument="oci", platform='pace')

In [ ]:
for item in results:
    summary = item.summary()
    print(summary["short-name"])

<div class="alert alert-info" role="alert">
The short name can also be found on <a href="https://search.earthdata.nasa.gov/search?fi=SPEXone!HARP2!OCI" target="_blank"> Eartdata Search</a>, directly under the collection name, after clicking on the "i" button for a collection in any search result.
</div>

Next, we use the `search_data` function to find granules within a
collection. Let's use the `short_name` for the PACE/OCI Level-2 near real time (NRT), product for biogeochemical properties (although you can
search for granules accross collections too).



The `count` argument limits the number of granules whose metadata is returned and stored in the `results` list.

In [ ]:
results = earthaccess.search_data(
    short_name="PACE_OCI_L2_BGC_NRT",
    count=1,
)

In [ ]:
results[0]

We can refine our search by passing more parameters that describe
the spatiotemporal domain of our use case. Here, we use the
`temporal` parameter to request a date range and the `bounding_box`
parameter to request granules that intersect with a bounding box. We
can even provide a `cloud_cover` threshold to limit files that have
a lower percetnage of cloud cover. We do not provide a `count`, so
we'll get all granules that satisfy the constraints.

Note, bbox is: [Minimum Longitude, Minimum Latitude, Maximum Longitude, Maximum Latitude]

In [ ]:
tspan = ("2026-09-09", "2026-09-09")
bbox = (16.0248, 42.6178, 16.3783, 42.925) 
clouds = (0, 60)

In [ ]:
results = earthaccess.search_data(
    short_name="PACE_OCI_L2_BGC_NRT",
    temporal=tspan,
    bounding_box=bbox,
    cloud_cover=clouds,
)

In [ ]:
len(results)

In [ ]:
results[0]

## 4. Open L2 Data

Let's go ahead and open a couple granules using `xarray`. The `earthaccess.open` function is used when you want to directly read bytes from a remote filesystem, but not download a whole file. When
running code on a host with direct access to the NASA Earthdata
Cloud, you don't need to download the data and `earthaccess.open`
is the way to go.

In [ ]:
paths = earthaccess.open(results)

The `paths` list contains references to files on a remote filesystem. The ob-cumulus-prod-public is the S3 Bucket in AWS us-west-2 region.

In [ ]:
dataset = xr.open_dataset(paths[0])
dataset

Notice that this `xarray.Dataset` has nothing but "Attributes". The NetCDF data model includes multi-group hierarchies within a single file, where each group maps to an `xarray.Dataset`. The whole file maps to a `DataTree`, which we will only use lightly because the implementation in XArray remains under development.

In [ ]:
datatree = open_datatree(paths[0])
datatree

In [ ]:
dataset = xr.merge(datatree.to_dict().values())
dataset

Here let's play with the raw data with lines as the y axis and pixels as the x axis

In [ ]:
dataset["chlor_a"].plot(cmap="viridis", vmin=.1, vmax=2)
plt.show()

Let's set the coordinates and plot with latitude and longitude so we can project the data onto a grid.

In [ ]:
dataset = dataset.set_coords(("longitude", "latitude"))
dataset["chlor_a"].plot(x="longitude", y="latitude", cmap="viridis", vmin=.1, vmax=2)
plt.show()

And if we want to get fancy, we can add the coastline.

In [ ]:
fig = plt.figure(figsize=(9,5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.gridlines(draw_labels={"left": "y", "bottom": "x"})
dataset["chlor_a"].plot(x="longitude", y="latitude", cmap="viridis", vmin=0.1, vmax=2, ax=ax)

plt.show()

But really chla is typically lognormally distributed to let's use a log normal scale

In [ ]:
fig = plt.figure(figsize=(16,7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.gridlines(draw_labels={"left": "y", "bottom": "x"})
plot = dataset["chlor_a"].plot(x="longitude", y="latitude", cmap=cmocean.cm.haline,norm=LogNorm(vmin=.1, vmax=2), ax=ax)

ax.set_xlim(14,18) #(16.0248, 42.6178, 16.3783, 42.925) 
ax.set_ylim(41,45)

# plt.savefig('figs/SPLIT_chla.png',dpi=300,transparent=True)

But some of this data may be suspicious so let's check out the flags. They're available in the data here:

In [ ]:
dataset.l2_flags

And you can inspect all the current flags on https://oceancolor.gsfc.nasa.gov/resources/atbd/ocl2flags/ and they're listed below and put into a python dictionary.

In [ ]:
# List l2 flags, then build them into a dict
l2_flags_list = [
    "ATMFAIL", "LAND", "PRODWARN", "HIGLINT", "HILT", "HISATZEN", "COASTZ",
    "SPARE", "STRAYLIGHT", "CLDICE", "COCCOLITH", "TURBIDW", "HISOLZEN",
    "SPARE", "LOWLW", "CHLFAIL", "NAVWARN", "ABSAER", "SPARE", "MAXAERITER",
    "MODGLINT", "CHLWARN", "ATMWARN", "SPARE", "SEAICE", "NAVFAIL", "FILTER",
    "SPARE", "BOWTIEDEL", "HIPOL", "PRODFAIL", "SPARE"]

L2_FLAGS = {flag: 1 << idx for idx, flag in enumerate(l2_flags_list)}

# Bailey and Werdell 2006 exclusion criteria
EXCLUSION_FLAGS = ["LAND", "HIGLINT", "HILT", "STRAYLIGHT", "CLDICE",
                   "ATMFAIL", "LOWLW", "FILTER", "NAVFAIL", "NAVWARN"]

In [ ]:
  # Calculate the bitwise OR of all flags in EXCLUSION_FLAGS to get a mask
  exclude_mask = sum(L2_FLAGS[flag] for flag in EXCLUSION_FLAGS)

  # Create a boolean mask
  # True means the flag value does not contain any of the EXCLUSION_FLAGS
  valid_mask = np.bitwise_and(dataset.l2_flags, exclude_mask) == 0

In [ ]:
dataset['flag_mask'] = valid_mask

In [ ]:
fig = plt.figure(figsize=(16,7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.gridlines(draw_labels={"left": "y", "bottom": "x"})
plot = dataset["chlor_a"].where(dataset.flag_mask).plot(x="longitude", y="latitude", cmap=cmocean.cm.haline,norm=LogNorm(vmin=.1, vmax=2), ax=ax)

ax.set_xlim(14,18) #(16.0248, 42.6178, 16.3783, 42.925) 
ax.set_ylim(41,45)

plt.show()

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">

**Ideas to play with the data more:**

* **Explore Carbon Estimates:** The BGC suite includes POC (particulate organic carbon) and several other estimates of carbon in the ocean. Can you plot some of these parameters? 
* **Check Out Other PACE Products:** A complete list of PACE products and their associated L2/L3 suites can be found in the [PACE Data Products Table](https://pace.oceansciences.org/data_table.htm). 
* **Investigate Radiation and Atmosphere:** Try checking out the PAR (photosynthetically available radiation) suite, or explore one of the atmospheric suites to look at cloud types or aerosol loads over the area.

</div>


[back to top](#Contents)

## 5. Open L3M Data

Let's use `earthaccess` to open some L3 mapped IOP granules. We will use a new search filter available in earthaccess.search_data: the granule_name argument accepts strings with the "*" wildcard. We need this to distinguish daily ("DAY") from eight-day ("8D") composites, as well as to get the 0.1 degree resolution projections.

In [ ]:
tspan = ("2026-07-07", "2026-07-13")

results = earthaccess.search_data(
    short_name="PACE_OCI_L3M_IOP_NRT",
    temporal=tspan,
    granule_name="*.DAY.*.0p1deg.*",
)

paths = earthaccess.open(results)

Let's open the first file using `xarray`.

In [ ]:
dataset = xr.open_dataset(paths[0])
dataset

Becuase the L3M variables have lat and lon coordinates, it's possible to stack multiple granules along a new dimension that corresponds to time. Instead of xr.open_dataset, we use xr.open_mfdataset to create a single xarray.Dataset (the "mf" in open_mfdataset stands for multiple files) from an array of paths.

The paths list is sorted temporally by default, which means the shape of the paths array specifies the way we need to tile the files together into larger arrays. We specify combine="nested" to combine the files according to the shape of the array of files (or file-like objects), even though paths is not a "nested" list in this case. The concat_dim="date" argument generates a new dimension in the combined dataset, because "date" is not an existing dimension in the individual files.

In [ ]:
dataset = xr.open_mfdataset(
    paths,
    combine="nested",
    concat_dim="date",
)
dataset

A common reason to generate a single dataset from multiple, daily images is to create a composite. Compare the map from a single day ...

In [ ]:
fig, ax = plt.subplots(figsize=(12,7))
dataset["aph_442"].sel({"date": 0}).plot(cmap=cmocean.cm.haline, norm=LogNorm(vmin=.001, vmax=1))
plt.show()

... to a map of average values, skipping "NaN" values that result from clouds.

In [ ]:
fig, ax = plt.subplots(figsize=(12,7))
dataset["aph_442"].mean("date").plot(cmap=cmocean.cm.haline, norm=LogNorm(vmin=.01, vmax=1))
plt.show()

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">

**Ideas to play with the data more:**

* **Improve the Projection:** Can you make this map projection a bit nicer or more visually appealing?
* **Explore Other Parameters:** Can you plot some other parameters? A list of all PACE products and their associated L2/L3 suites can be found in the [PACE Data Products Table](https://pace.oceansciences.org/data_table.htm).
* **Analyze Variability:** Can you plot a measure of variability (such as standard deviation or variance) for a single parameter?

</div>

## 6. Download Data

Let's go ahead and download a couple granules.

Let's look at the `earthaccess.download` function, which is used
to copy files onto a filesystem local to the machine executing the
code. For this function, provide the output of
`earthaccess.search_data` along with a directory where `earthaccess` will store downloaded granules.

Even if you only want to read a slice of the data, and downloading
seems unncessary, if you use `earthaccess.open` while not running on a remote host with direct access to the NASA Earthdata Cloud,
performance will be very poor. This is not a problem with "the
cloud" or with `earthaccess`, it has to do with the data format and may soon be resolved.


<div class="alert alert-info" role="alert">
Note i have commented out this section to stop you from downloading gigantic datasets. But code is here for you to try it out on your own if you wish to do that later. 
</div>

In [ ]:

# results = earthaccess.search_data(
#     short_name="PACE_OCI_L2_BGC_NRT",
#     temporal=tspan,
#     bounding_box=bbox,
#     cloud_cover=clouds,
# )

The `paths` list now contains paths to actual files on the local
filesystem.

In [ ]:
# paths = earthaccess.download(results, local_path="data")
# paths

We can open up that locally saved file using `xarray` as well.

In [ ]:
# open_datatree(paths[0])

[back to top](#Contents)

# 7. Now let's look at some spectra

In [ ]:
tspan = ("2026-09-09", "2026-09-09")
bbox = (16.0248, 42.6178, 16.3783, 42.925) 
clouds = (0, 60)

We will take a L2 AOP suite file, to look at the actual ocean color data.

In [ ]:

results = earthaccess.search_data(
    short_name="PACE_OCI_L2_AOP_NRT",
    temporal=tspan,
    bounding_box=bbox,
    cloud_cover=clouds
)

paths = earthaccess.open(results)

In [ ]:
datatree = xr.open_datatree(paths[-1])
rrs = datatree["geophysical_data"]["Rrs"]
rrs

The Rrs variable has 172 values in the wavelength; the blue, red, and SWIR wavelengths have been combined. Without having to look too closely at the exact wavelength coordinates, we can plot the variable for a blue wavelength using method="nearest".

In [ ]:
plot = rrs.sel({"wavelength": 440}, method="nearest").plot(cmap="viridis", robust=True)

The scene is being plotted using number_of_lines and pixels_per_line as “x” and “y”, respectively. We need to add more coordinates, the latitude and longitude, for a true map. These coordinates variables are in the “navigation_data” group.

Now we often want to grab a specific pixel and inspect the spectra and other attributes

In [ ]:
for item in ("longitude", "latitude"):
    rrs[item] = datatree["navigation_data"][item]
rrs


In [ ]:
rrs_sel = rrs.sel({"wavelength": 440}, method="nearest")
plot = rrs_sel.plot(x="longitude", y="latitude", cmap="viridis", robust=True)

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
im = rrs_sel.plot(x="longitude", y="latitude", cmap="viridis", robust=True, ax=ax)
ax.gridlines(draw_labels={"left": "y", "bottom": "x"})
ax.coastlines()
plt.show()

Let’s plot the full “Rrs” spectrum for individual pixels. A visualization with all the pixels wouldn’t be useful, but limiting to a bounding box gives a simple way to subset pixels. Here we define a center lat/lon, and take 5X5 pixels around it. In this case i am using Stoncica Bay 43 N,  N 16.33 E. 

In [ ]:
# Define central point #43.0, 16.333333
center_lat = 43
center_lon = 16.333

# Compute squared distance from every pixel to the center
# (works fine for finding the nearest pixel over a small area)
dist2 = (rrs["latitude"] - center_lat) ** 2 + (rrs["longitude"] - center_lon) ** 2

# Find the index of the closest pixel (in 2D: number_of_lines, pixels_per_line)
min_idx = np.unravel_index(np.nanargmin(dist2.values), dist2.shape)
line_c, pix_c = int(min_idx[0]), int(min_idx[1])


# Define a 5x5 window (2 pixels on each side of the center) you can change this to be 3X3 window
half = 2  # 5x5 box

# Clip to array bounds so we don't go out of range at edges
n_lines = rrs.sizes["number_of_lines"]
n_pix = rrs.sizes["pixels_per_line"]

line_start = max(line_c - half, 0)
line_stop = min(line_c + half + 1, n_lines)   # +1 because slice stop is exclusive
pix_start = max(pix_c - half, 0)
pix_stop = min(pix_c + half + 1, n_pix)

# Select the 5x5 box using integer-position indexing
rrs_box = rrs.isel(
    number_of_lines=slice(line_start, line_stop),
    pixels_per_line=slice(pix_start, pix_stop),
)

print(rrs_box.sizes)

The line plotting method will only draw a line plot for 1D data, which we can get by stacking our two spatial dimensions and choosing to show the new “pixel dimension” in purple, and adding a dark line as an average of Rrs in our selected pixel group.

In [ ]:
# Stack the spatial dimensions into a single "pixel" dimension
rrs_stack = rrs_box.stack(
    {"pixel": ["number_of_lines", "pixels_per_line"]},
    create_index=False,
)

plot = rrs_stack.plot(
    hue="pixel",
    add_legend=False,
    color="plum",
    alpha=0.4,
    linewidth=0.8,
)

rrs_median = rrs_stack.median(dim="pixel", skipna=True)

# Overlay the median as a thick black line
rrs_median.plot(
    color="black",
    linewidth=2.5,
    label="Median Rrs",
)

plt.legend()
plt.tight_layout()
plt.show()

If you want to, now you can export this data for your further analysis. 

In [ ]:

df = rrs_stack.to_dataframe(name="Rrs").reset_index()

# 2. Reshape so each pixel is a row and wavelengths are columns
csv_df = df.pivot(
    index=["pixel", "latitude", "longitude"], 
    columns="wavelength", 
    values="Rrs"
).reset_index()

# 3. Clean up: Drop the arbitrary pixel index and strip any empty wavelength columns
csv_df = csv_df.drop(columns=["pixel"]).dropna(axis=1, how="all")

# Save to your file
csv_df.to_csv("test_station_only.csv", index=False)



<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">

**Ideas to play with the data more:**

* **Map Other Wavelengths:** How does the distribution of other colors differ from 440 nm? *(Note: Variability in the blue band (440 nm) is heavily influenced by chlorophyll absorption and the absorption of colored dissolved organic matter (CDOM).)*
* **Plot Different Regions:** Can you make a figure of spectra from another area? Try looking at the Black Sea or close to the mouth of the Nile River.
* **Analyze Different Dates:** Can you generate this figure for another day, such as September 16th? Alternatively, pick another clear and pretty day using [NASA Worldview](https://worldview.earthdata.nasa.gov/) as shown above.

</div>

# 7. Now let's look do a cutesy time series analysis for Stoncica Bay

In their [paper from 2018](https://www.mdpi.com/2072-4292/10/9/1460), Kovac et al. have used 55-year long dataset from Stoncica Bay to evaluate seasonality and regime change of phytoplankton and primary productivity in Stoncica Bay. In his awesome graphic abstract, they compare in situ vs. remote sensing seasonality. Lets check if 2025 was the same or different than their published seasonal pattern. Usually, one would download the data, but I couldn't find it so here we used the the cheat way (e.g., AI) to read the values on the graph. 

Cool - lets get some chlorophyll data from PACE. We will grab monthly L3m data here and superimpose it on top of the data from Kovac et al. graph. 

In [ ]:
tspan = ("2025-01", "2025-12")
results_BGC = earthaccess.search_data(
    short_name="PACE_OCI_L3M_BGC",
    granule_name="*.MO.*4km*",
    temporal=tspan
)

In [ ]:
path_files = earthaccess.open(results_BGC)

Because we want to create a timeline, we will extract the date information from the dataset attributes using this `time_from_attr` function.

In [ ]:
def time_from_attr(ds):
    """Set the time attribute as a dataset variable
    Args:
        ds: a dataset corresponding to one or multiple Level-2 granules
    Returns:
        the dataset with a scalar "time" coordinate
    """
    datetime = ds.attrs["time_coverage_start"].replace("Z", "")
    ds["date"] = ((), np.datetime64(datetime, "ns"))
    ds = ds.set_coords("date")
    return ds

We use `time_from_attr` as the `preprocess` function in `xr.open_mfdataset`. This extracts the date information from each file’s attributes, allowing the date to be added as a coordinate to the resulting dataset.

In [ ]:
dataset_BGC = xr.open_mfdataset(
    path_files, preprocess=time_from_attr, combine="nested", concat_dim="date"
)
dataset_BGC

Let's crop the data to the bounding box we used before to zoom into the Stoncica Bay area. The site is at 43.0, 16.333333, so we will take a tiny bounding box (3x3) around it. 

In [ ]:
# Target coordinates for the center pixel
target_lon = 16.333333
target_lat = 43.0

# 1. Find the nearest array indices for the target coordinates
lon_idx = np.abs(dataset_BGC.lon.values - target_lon).argmin()
lat_idx = np.abs(dataset_BGC.lat.values - target_lat).argmin()

# 2. Crop exactly 3x3 pixels using .isel (index selection)
# slice(idx - 1, idx + 2) grabs exactly 3 items: [idx-1, idx, idx+1]
dataset_crop = dataset_BGC.isel(
    lon=slice(lon_idx - 1, lon_idx + 2),
    lat=slice(lat_idx - 1, lat_idx + 2)
)

# Display the resulting 3x3 dataset
dataset_crop

ok - lets see if the bounding box is in the correct spot. 

In [ ]:
chl_data = dataset_crop["chlor_a"].isel(date=5)

box_lon_min = dataset_crop.lon.min().item()
box_lon_max = dataset_crop.lon.max().item()
box_lat_min = dataset_crop.lat.min().item()
box_lat_max = dataset_crop.lat.max().item()

# 1. Define the new map boundaries: exactly 1 degrees outward on each side
map_lon_min = box_lon_min - 1.0
map_lon_max = box_lon_max + 1.0
map_lat_min = box_lat_min - 1.0
map_lat_max = box_lat_max + 1.0

# 2. Extract the chlor_a data for this new 2-degree expanded area 
# We pull from dataset_moana (the full dataset) so the data reaches the new edges
chl_data = dataset_BGC["chlor_a"].isel(date=5).sel(
    lon=slice(map_lon_min, map_lon_max), 
    lat=slice(map_lat_max, map_lat_min) # Ensure lat is sliced max to min if descending
)

# 3. Create the plot
fig, ax = plt.subplots(figsize=(8, 5), subplot_kw={"projection": ccrs.PlateCarree()})

# Plot chlor_a using pcolormesh
# We use LogNorm because chlorophyll values usually span multiple orders of magnitude
mesh = ax.pcolormesh(
    chl_data.lon, 
    chl_data.lat, 
    chl_data.values, 
    transform=ccrs.PlateCarree(),
    cmap="viridis",  # Standard colormap for chlorophyll
    norm=LogNorm(vmin=0.1, vmax=1)
)

# Add a colorbar
cbar = plt.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.8, pad=0.05)
cbar.set_label("Chlorophyll-a [mg m^-3]")

# Add map features
ax.coastlines(linewidth=0.6, color="grey", zorder=3)
gl = ax.gridlines(
    draw_labels={"bottom": True, "left": True, "top": False, "right": False},
    linewidth=0.5,
    alpha=0.5,
    linestyle="--",
)

# Draw the red bounding box around the 3x3 pixel area
ax.add_patch(
    Rectangle(
        (box_lon_min, box_lat_min),
        box_lon_max - box_lon_min,
        box_lat_max - box_lat_min,
        edgecolor="red",
        facecolor="none",
        linewidth=2,
        transform=ccrs.PlateCarree(),
        zorder=4 # Ensure it plots on top of the data
    )
)

# Set the map extent to match your wider surrounding area crop
ax.set_extent([
    map_lon_min, 
    map_lon_max, 
    map_lat_min, 
    map_lat_max
], crs=ccrs.PlateCarree())

date_str = dataset_crop.date.isel(date=5).dt.strftime("%Y-%m-%d").item()
#plt.title(f"Chlorophyll-a and 3x3 Target Box - {date_str}", fontsize=12)

plt.tight_layout()
plt.show()

Let us calculate the median value for Chlorophyll for our box, for each month of 2025. 

In [ ]:
chlor_a_monthly = dataset_crop["chlor_a"].median(dim=["lat", "lon"])

In [ ]:
chlor_a_monthly

And now we used the scrapped from the paper and data from PACE. 

In [ ]:

# Combined dataset extracted from both graphs, Figure 1
data = {
    "Month": [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun", 
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
    ],
    "Chlorophyll": [
        0.225, 0.220, 0.204, 0.186, 0.162, 0.117, 
        0.123, 0.123, 0.121, 0.156, 0.199, 0.228,
    ],
    "PzT_InSitu": [
        130, 140, 210, 250, 270, 230, 
        160, 135, 140, 155, 150, 135,
    ],
    "PzT_climatological_OC": [
        215, 260, 290, 340, 370, 340, 
        285, 225, 190, 175, 185, 210,
    ],
}

# Create DataFrame
df = pd.DataFrame(data)
df.set_index("Month", inplace=True)

# Quick visualization check
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot Pz,T curves (left y-axis)
ax1.set_xlabel("Month")
ax1.set_ylabel(r"$P_{z,T}\ [mg\ C\ m^{-2}]$", color="darkorange")
ax1.plot(df.index, df["PzT_InSitu"], color="darkorange", linewidth=2.5, label="$P_{z,T}$ (PzT_InSitu)")
ax1.plot(df.index, df["PzT_climatological_OC"], color="lightgray", linewidth=2, label="PzT_climatological_OC")
ax1.tick_params(axis="y", labelcolor="dimgray")

# Plot Biomass B dots (right y-axis)
ax2 = ax1.twinx()
ax2.set_ylabel(r"$Chlorophyll\ [mg\ Chl\ m^{-3}]$", color="forestgreen")
ax2.scatter(df.index, df["Chlorophyll"], color="forestgreen", s=60, label="Chlorophyll Dots")

# ====================================================================
# --- ADDED: Plot the satellite chlor_a_ts on this axis ---
# We use .values to drop the xarray index so it easily plots over the "Jan", "Feb" strings
ax2.plot(
    df.index, 
    chlor_a_monthly.values, 
    color="limegreen", 
    linewidth=2.5, 
    linestyle="--", 
    label="PACE chlor_a 2025 (3x3 Box)"
)
# ====================================================================

ax2.tick_params(axis="y", labelcolor="forestgreen")

# Extract handles and labels from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

# Create a single legend and position it outside the axes on the right
ax2.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper left", bbox_to_anchor=(1.15, 1))

plt.title("Reconstructed Chart Preview from Kovac et al 2018 with Satellite Data")

plt.tight_layout()
plt.show()

<div style="background-color: #EDE7F6; padding: 20px; border-radius: 8px; color: #333333;">

**Ideas to play with the data more:**

* **Check 2026 Seasonality:** It seems that 2025 was similar to the established chlorophyll seasonality (small differences might be attributable to the different data inputs; they used the OC-CCI dataset, whereas we used PACE here). Can you try to add available data from 2026 to see if it fits the established seasonality?
* **Superimpose PACE NPP:** PACE has a primary productivity product, `NPP_cafe`, that uses a different model based on a publication by Silsbe et al. (2016). Check out the [Earthdata NPP_cafe documentation here](https://www.earthdata.nasa.gov/data/catalog/ob-cloud-pace-oci-l4m-npp-cafe-3.2-0). Can you use the code above to superimpose the productivity from PACE onto the published data?

</div>